# SAC with RGBD Observations — PushCube-v1

Trains SAC on `PushCube-v1` using RGB camera observations with WandB logging.

Equivalent CLI:
```bash
python sac_rgbd.py --env_id="PushCube-v1" --obs_mode="rgb" \
  --num_envs=32 --utd=0.5 --buffer_size=300_000 \
  --control_mode="pd_ee_delta_pos" --camera_width=64 --camera_height=64 \
  --total_timesteps=1_000_000 --eval_freq=10_000 --track
```

## Installation

In [ ]:
# @title Setup Vulkan + dependencies
!mkdir -p /usr/share/vulkan/icd.d
!wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/nvidia_icd.json
!wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/10_nvidia.json
!mv nvidia_icd.json /usr/share/vulkan/icd.d
!mv 10_nvidia.json /usr/share/glvnd/egl_vendor.d/10_nvidia.json
!apt-get install -y --no-install-recommends libvulkan-dev
!pip install --upgrade mani_skill tyro wandb
# fetch the upstream SAC-RGBD script so we can import from it
!wget -q -O sac_rgbd.py https://raw.githubusercontent.com/haosulab/ManiSkill/main/examples/baselines/sac/sac_rgbd.py

In [ ]:
# @title WandB login
import wandb
wandb.login()

## Imports from upstream sac_rgbd.py

In [ ]:
import os
os.environ["MANI_SKILL_DATA_DIR"] = "/root/.maniskill/data"

# stdlib / third-party
from collections import defaultdict
import random, time
import tqdm
import gymnasium as gym
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import wandb

# ManiSkill wrappers
from mani_skill.utils import gym_utils
from mani_skill.utils.wrappers.flatten import FlattenActionSpaceWrapper, FlattenRGBDObservationWrapper
from mani_skill.utils.wrappers.record import RecordEpisode
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv
import mani_skill.envs

# models + replay buffer from upstream script
import sac_rgbd
sac_rgbd.wandb = wandb  # sac_rgbd only imports wandb inside __main__; inject it here
from sac_rgbd import Actor, SoftQNetwork, ReplayBuffer, Logger

## Configuration

In [ ]:
# ── Edit these to change the run ─────────────────────────────────────────────
ENV_ID          = "PushCube-v1"
OBS_MODE        = "rgb"              # "rgb", "rgbd", or "depth"
INCLUDE_STATE   = True
CONTROL_MODE    = "pd_ee_delta_pos"
CAMERA_WIDTH    = 64
CAMERA_HEIGHT   = 64

NUM_ENVS        = 8
NUM_EVAL_ENVS   = 16
TOTAL_TIMESTEPS = 1_000_000
BUFFER_SIZE     = 300_000
BATCH_SIZE      = 512
LEARNING_STARTS = 4_000
UTD             = 0.5
TRAINING_FREQ   = 64
EVAL_FREQ       = 10_000
NUM_EVAL_STEPS  = 50
LOG_FREQ        = 1_000
VIDEO_LOG_FREQ  = 50_000            # log one eval video to WandB every N env steps

GAMMA           = 0.8
TAU             = 0.01
POLICY_LR       = 3e-4
Q_LR            = 3e-4
AUTOTUNE        = True
ALPHA           = 0.2
BUFFER_DEVICE   = "cuda"
SEED            = 1
CAPTURE_VIDEO   = True
SAVE_MODEL      = True

TRACK           = True               # False to disable WandB
WANDB_PROJECT   = "ManiSkill"
WANDB_ENTITY    = None
WANDB_GROUP     = "SAC"

# derived
GRAD_STEPS      = int(TRAINING_FREQ * UTD)
STEPS_PER_ENV   = TRAINING_FREQ // NUM_ENVS
RUN_NAME        = f"{ENV_ID}__sac_rgbd__{SEED}__{int(time.time())}"
print(f"Run: {RUN_NAME}  |  grad_steps/iter: {GRAD_STEPS}  |  steps_per_env: {STEPS_PER_ENV}")

## Environment Setup

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

env_kwargs = dict(
    obs_mode=OBS_MODE, render_mode="all", sim_backend="gpu",
    control_mode=CONTROL_MODE,
    sensor_configs=dict(width=CAMERA_WIDTH, height=CAMERA_HEIGHT),
)

envs      = gym.make(ENV_ID, num_envs=NUM_ENVS, **env_kwargs)
eval_envs = gym.make(ENV_ID, num_envs=NUM_EVAL_ENVS,
                     human_render_camera_configs=dict(shader_pack="default"), **env_kwargs)

use_rgb   = OBS_MODE in ("rgb", "rgbd")
use_depth = OBS_MODE in ("depth", "rgbd")
envs      = FlattenRGBDObservationWrapper(envs,      rgb=use_rgb, depth=use_depth, state=INCLUDE_STATE)
eval_envs = FlattenRGBDObservationWrapper(eval_envs, rgb=use_rgb, depth=use_depth, state=INCLUDE_STATE)

if isinstance(envs.action_space, gym.spaces.Dict):
    envs      = FlattenActionSpaceWrapper(envs)
    eval_envs = FlattenActionSpaceWrapper(eval_envs)

eval_output_dir = f"runs/{RUN_NAME}/videos"
eval_envs = RecordEpisode(eval_envs, output_dir=eval_output_dir,
                          save_trajectory=False, save_video=CAPTURE_VIDEO,
                          trajectory_name="trajectory",
                          max_steps_per_video=NUM_EVAL_STEPS, video_fps=30)

envs      = ManiSkillVectorEnv(envs,      NUM_ENVS,      ignore_terminations=True, record_metrics=True)
eval_envs = ManiSkillVectorEnv(eval_envs, NUM_EVAL_ENVS, ignore_terminations=True, record_metrics=True)
assert isinstance(envs.single_action_space, gym.spaces.Box)

max_episode_steps = gym_utils.find_max_episode_steps_value(envs._env)
print(f"Max episode steps: {max_episode_steps}")

## Initialize Networks & WandB

In [ ]:
obs, _       = envs.reset(seed=SEED)
eval_obs, _  = eval_envs.reset(seed=SEED)

actor   = Actor(envs, sample_obs=obs).to(device)
qf1     = SoftQNetwork(envs, actor.encoder).to(device)
qf2     = SoftQNetwork(envs, actor.encoder).to(device)
qf1_tgt = SoftQNetwork(envs, actor.encoder).to(device)
qf2_tgt = SoftQNetwork(envs, actor.encoder).to(device)
qf1_tgt.load_state_dict(qf1.state_dict())
qf2_tgt.load_state_dict(qf2.state_dict())

q_optimizer     = optim.Adam(
    list(qf1.mlp.parameters()) + list(qf2.mlp.parameters()) + list(qf1.encoder.parameters()),
    lr=Q_LR)
actor_optimizer = optim.Adam(list(actor.parameters()), lr=POLICY_LR)

if AUTOTUNE:
    target_entropy = -torch.prod(torch.Tensor(envs.single_action_space.shape).to(device)).item()
    log_alpha      = torch.zeros(1, requires_grad=True, device=device)
    alpha          = log_alpha.exp().item()
    a_optimizer    = optim.Adam([log_alpha], lr=Q_LR)
else:
    alpha = ALPHA

config = dict(
    env_id=ENV_ID, obs_mode=OBS_MODE, control_mode=CONTROL_MODE,
    camera_width=CAMERA_WIDTH, camera_height=CAMERA_HEIGHT,
    num_envs=NUM_ENVS, total_timesteps=TOTAL_TIMESTEPS,
    buffer_size=BUFFER_SIZE, batch_size=BATCH_SIZE,
    learning_starts=LEARNING_STARTS, utd=UTD,
    gamma=GAMMA, tau=TAU, policy_lr=POLICY_LR, q_lr=Q_LR,
    autotune=AUTOTUNE, seed=SEED, env_horizon=max_episode_steps,
)
if TRACK:
    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, sync_tensorboard=False,
               config=config, name=RUN_NAME, save_code=True,
               group=WANDB_GROUP, tags=["sac", "rgbd"])

writer = SummaryWriter(f"runs/{RUN_NAME}")
logger = Logger(log_wandb=TRACK, tensorboard=writer)

envs.single_observation_space.dtype = np.float32
rb = ReplayBuffer(env=envs, num_envs=NUM_ENVS, buffer_size=BUFFER_SIZE,
                  storage_device=torch.device(BUFFER_DEVICE), sample_device=device)
print(f"Actor params: {sum(p.numel() for p in actor.parameters()):,}")

## Training Loop

In [ ]:
import glob as _glob

global_step          = 0
global_update        = 0
learning_has_started = False
cumulative_times     = defaultdict(float)
global_steps_per_iter = NUM_ENVS * STEPS_PER_ENV
pbar = tqdm.tqdm(total=TOTAL_TIMESTEPS, desc="Training")

while global_step < TOTAL_TIMESTEPS:

    # ── Evaluation ────────────────────────────────────────────────────────
    if EVAL_FREQ > 0 and (global_step - TRAINING_FREQ) // EVAL_FREQ < global_step // EVAL_FREQ:
        actor.eval()
        stime = time.perf_counter()
        eval_obs, _ = eval_envs.reset()
        eval_metrics = defaultdict(list)
        for _ in range(NUM_EVAL_STEPS):
            with torch.no_grad():
                eval_obs, _, _, _, eval_infos = eval_envs.step(actor.get_eval_action(eval_obs))
            if "final_info" in eval_infos:
                for k, v in eval_infos["final_info"]["episode"].items():
                    eval_metrics[k].append(v)
        eval_means = {k: torch.stack(v).float().mean() for k, v in eval_metrics.items()}
        for k, v in eval_means.items():
            logger.add_scalar(f"eval/{k}", v, global_step)
        eval_time = time.perf_counter() - stime
        cumulative_times["eval_time"] += eval_time
        logger.add_scalar("time/eval_time", eval_time, global_step)
        pbar.set_postfix(
            success=f"{eval_means.get('success_once', torch.tensor(0)):.2f}",
            ret=f"{eval_means.get('return', torch.tensor(0)):.2f}",
        )
        actor.train()

        # log one eval video to WandB at VIDEO_LOG_FREQ intervals
        if TRACK and VIDEO_LOG_FREQ > 0 and (global_step - TRAINING_FREQ) // VIDEO_LOG_FREQ < global_step // VIDEO_LOG_FREQ:
            videos = sorted(_glob.glob(f"{eval_output_dir}/*.mp4"))
            if videos:
                wandb.log({"eval/video": wandb.Video(videos[-1], fps=30, format="mp4")}, step=global_step)

        if SAVE_MODEL:
            torch.save({"actor": actor.state_dict(), "qf1": qf1_tgt.state_dict(),
                        "qf2": qf2_tgt.state_dict(),
                        "log_alpha": log_alpha if AUTOTUNE else None},
                       f"runs/{RUN_NAME}/ckpt_{global_step}.pt")

    # ── Rollout ────────────────────────────────────────────────────────────
    t_rollout = time.perf_counter()
    for _ in range(STEPS_PER_ENV):
        global_step += NUM_ENVS
        if not learning_has_started:
            actions = 2 * torch.rand(envs.action_space.shape, device=device) - 1
        else:
            with torch.no_grad():
                actions, _, _, _ = actor.get_action(obs)
        next_obs, rewards, terminations, truncations, infos = envs.step(actions)

        # per-step reward: mean across all parallel envs at this timestep
        logger.add_scalar("train/reward", rewards.mean().item(), global_step)

        real_next_obs = {k: v.clone() for k, v in next_obs.items()}
        need_final_obs = truncations | terminations   # bootstrap_at_done="always"
        stop_bootstrap = torch.zeros_like(terminations, dtype=torch.bool)
        if "final_info" in infos:
            for k in real_next_obs:
                real_next_obs[k][need_final_obs] = infos["final_observation"][k][need_final_obs].clone()
            done_mask = infos["_final_info"]
            for k, v in infos["final_info"]["episode"].items():
                logger.add_scalar(f"train/{k}", v[done_mask].float().mean(), global_step)
        rb.add(obs, real_next_obs, actions, rewards, stop_bootstrap)
        obs = next_obs
    rollout_time = time.perf_counter() - t_rollout
    cumulative_times["rollout_time"] += rollout_time
    pbar.update(NUM_ENVS * STEPS_PER_ENV)

    if global_step < LEARNING_STARTS:
        continue

    # ── Gradient updates ───────────────────────────────────────────────────
    t_update = time.perf_counter()
    learning_has_started = True
    for _ in range(GRAD_STEPS):
        global_update += 1
        data = rb.sample(BATCH_SIZE)
        o, no, act, rew, done = data.obs, data.next_obs, data.actions, data.rewards, data.dones

        with torch.no_grad():
            na, nlogpi, _, vf = actor.get_action(no)
            next_q = rew.flatten() + (1 - done.flatten()) * GAMMA * (
                torch.min(qf1_tgt(no, na, vf), qf2_tgt(no, na, vf)) - alpha * nlogpi).view(-1)

        vf_obs  = actor.encoder(o)
        q1_val  = qf1(o, act, vf_obs).view(-1)
        q2_val  = qf2(o, act, vf_obs).view(-1)
        qf1_loss = F.mse_loss(q1_val, next_q)
        qf2_loss = F.mse_loss(q2_val, next_q)
        q_optimizer.zero_grad()
        (qf1_loss + qf2_loss).backward()
        q_optimizer.step()

        pi, log_pi, _, vf_obs = actor.get_action(o)
        actor_loss = ((alpha * log_pi) - torch.min(
            qf1(o, pi, vf_obs, detach_encoder=True),
            qf2(o, pi, vf_obs, detach_encoder=True)).view(-1)).mean()
        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        if AUTOTUNE:
            with torch.no_grad():
                _, log_pi, _, _ = actor.get_action(o)
            alpha_loss = (-log_alpha.exp() * (log_pi + target_entropy)).mean()
            a_optimizer.zero_grad()
            alpha_loss.backward()
            a_optimizer.step()
            alpha = log_alpha.exp().item()

        for p, tp in zip(qf1.parameters(), qf1_tgt.parameters()):
            tp.data.copy_(TAU * p.data + (1 - TAU) * tp.data)
        for p, tp in zip(qf2.parameters(), qf2_tgt.parameters()):
            tp.data.copy_(TAU * p.data + (1 - TAU) * tp.data)

    update_time = time.perf_counter() - t_update
    cumulative_times["update_time"] += update_time

    # ── Logging ────────────────────────────────────────────────────────────
    if (global_step - TRAINING_FREQ) // LOG_FREQ < global_step // LOG_FREQ:
        for tag, val in [
            ("losses/qf1_values", q1_val.mean().item()),
            ("losses/qf2_values", q2_val.mean().item()),
            ("losses/qf1_loss",   qf1_loss.item()),
            ("losses/qf2_loss",   qf2_loss.item()),
            ("losses/qf_loss",    (qf1_loss + qf2_loss).item() / 2),
            ("losses/actor_loss", actor_loss.item()),
            ("losses/alpha",      alpha),
            ("time/update_time",  update_time),
            ("time/rollout_time", rollout_time),
            ("time/rollout_fps",  global_steps_per_iter / rollout_time),
        ]:
            logger.add_scalar(tag, val, global_step)
        for k, v in cumulative_times.items():
            logger.add_scalar(f"time/total_{k}", v, global_step)
        if AUTOTUNE:
            logger.add_scalar("losses/alpha_loss", alpha_loss.item(), global_step)

pbar.close()

if SAVE_MODEL:
    model_path = f"runs/{RUN_NAME}/final_ckpt.pt"
    torch.save({"actor": actor.state_dict(), "qf1": qf1_tgt.state_dict(),
                "qf2": qf2_tgt.state_dict(),
                "log_alpha": log_alpha if AUTOTUNE else None}, model_path)
    print(f"Model saved to {model_path}")

logger.close()
if TRACK:
    wandb.finish()
envs.close()
eval_envs.close()

## Show Evaluation Video

In [ ]:
import glob
from IPython.display import Video

videos = sorted(glob.glob(f"runs/{RUN_NAME}/videos/*.mp4"))
Video(videos[-1], embed=True, width=640) if videos else print("No videos found.")